# Training on INCLUDE_50 Dataset

## Preprocessing Data

In [57]:
import pandas as pd
import numpy as np
import os

import tensorflow as tf
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Dense, LSTM, Bidirectional, Flatten, TimeDistributed, Reshape
from tensorflow.keras.models import Model

from sklearn.model_selection import train_test_split

from tqdm.notebook import tqdm

In [58]:
# Checking whether the gpu is available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        print("Using GPU:", gpus)

    except RuntimeError as e:
        print(e)

else:
    print("No GPU found.")

Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [59]:
train_data = pd.read_csv('Dataset/train.csv')
test_data = pd.read_csv('Dataset/test.csv')

train_data.sample(5)

,parent_label,label,video_path,include_50
1973,Home,48. Card,Home/48. Card/MVI_8854.MP4,False
2000,Society,1. Religion,Society/1. Religion/MVI_8657.MP4,False
3655,Means_of_Transportation,11. Car,Means_of_Transportation/11. Car/MVI_3117.mp4,True
3190,Seasons,64. Fall,Seasons/64. Fall/MVI_4580.mp4,True
906,Days_and_Time,76. Week,Days_and_Time/76. Week/MVI_9170.MP4,False


In [ ]:
# labels = train_data['label']
# train_data_path = train_data['vedio_path']

# Extracting labels and video paths for the videos that are from the include 50 dataset

labels = []
train_path_I50 = []

for i in range(len(train_data)):
    if train_data['include_50'][i] == True:
        labels.append(train_data['label'][i])
        train_path_I50.append("Dataset\\" + train_data['video_path'][i])
        
labels = pd.Series(labels).unique()
labels = pd.Series(labels).to_list()

train_path_I50 = pd.Series(train_path_I50)

# train_path_I50.sample(5), labels.sample(5)
labels[:5]

['1. loud', '19. House', '83. big large', '91. Priest', '23. Court']

In [61]:
label_map = dict()

# Creating a dictionary of labels with their corresponding index 
for i in range(len(labels)):
    # Splitting Label from num
    split = labels[i].split(" ")
    
    # Joining the ramining str to make a full label name
    label = split[1:]
    label = " ".join(label)
        
    label_map[label] = i

labels = list(label_map.keys())
  
label_map    

{'loud': 0,
 'House': 1,
 'big large': 2,
 'Priest': 3,
 'Court': 4,
 'train ticket': 5,
 'it': 6,
 'Shoes': 7,
 'Dog': 8,
 'Bank': 9,
 'Thank you': 10,
 'Election': 11,
 'Cow': 12,
 'Window': 13,
 'quiet': 14,
 'dry': 15,
 'long': 16,
 'Hello': 17,
 'Bird': 18,
 'Hat': 19,
 'Black': 20,
 'short': 21,
 'White': 22,
 'Fan': 23,
 'new': 24,
 'Store or Shop': 25,
 'Monday': 26,
 'Death': 27,
 'Cell phone': 28,
 'you (plural)': 29,
 'T-Shirt': 30,
 'Girl': 31,
 'Father': 32,
 'Red': 33,
 'hot': 34,
 'Fall': 35,
 'I': 36,
 'Time': 37,
 'Car': 38,
 'Good Morning': 39,
 'Summer': 40,
 'Paint': 41,
 'Teacher': 42,
 'Brother': 43,
 'good': 44,
 'happy': 45,
 'Boy': 46,
 'small little': 47,
 'Pen': 48,
 'Year': 49}

In [75]:
# Loading all the labeled videos in the dataset
X = []
y= []

for label in tqdm(labels):
    label_videos = os.listdir("MP_data/"+label)
    # window = []
    
    for video in label_videos:        
        res = np.load("MP_data/" + f"{label}/" + video,allow_pickle=True)
        X.append(res)
        y.append(label_map[label])        

len(X),len(y)

        

  0%|          | 0/50 [00:00<?, ?it/s]

(675, 675)

In [76]:
# X,y = np.array(X),np.array(y)
# X.shape,y.shape
len(X[0])

66

In [ ]:
# Bhai yha meine kya hi harkat kari thi gandi wali bhai agar meine har vedio ka input ek baar mein hi pahucha dunga toh kaise kaam banega sab vedio ke inputs ko alag lena hoga na bhai :))))))

# Padding the videos to make them of the same length

max_frames = max([len(video) for video in X])
max_frames

X = pad_sequences(X, maxlen=max_frames, padding='post', dtype='float32')

print(X.shape)


(675, 154, 1662)


In [91]:
X= np.array(X)
y= np.array(y)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42,shuffle=True) 

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)


X_train shape: (540, 154, 1662)
X_val shape: (135, 154, 1662)
y_train shape: (540,)
y_val shape: (135,)


# Model

## Architecture

In [85]:
import keras

# input_shape = X_train[0].shape #(19, 19, 1662)
input_shape = (154, 1662)
num_classes =  len(label_map.keys())#50
# batch_size = 50

INCLUDESEQ50_V2 = keras.Sequential([
        
        Input(shape=input_shape),        
        # Input(shape=input_shape),

        # This TimeDistributed layer applies the FLatten layer to each time step
        
        # TimeDistributed(Flatten(), input_shape=input_shape),
        
        # Bidirectional LSTM layers
        Bidirectional(LSTM(64, return_sequences=True)),
        Bidirectional(LSTM(64, return_sequences=True)),
        Bidirectional(LSTM(64, return_sequences=True)),
        Bidirectional(LSTM(64, return_sequences=True)),
        
        # Flatten the output
        Flatten(),
        
        # Fully connected layer
        Dense(128, activation='relu'),
        
        # Output layer
        Dense(num_classes, activation='softmax')
])

model = INCLUDESEQ50_V2

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
 
# model.build(input_shape=(None, *input_shape))
model.build(input_shape=(input_shape))

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bidirectional (Bidirectiona  (None, 154, 128)         884224    
 l)                                                              
                                                                 
 bidirectional_1 (Bidirectio  (None, 154, 128)         98816     
 nal)                                                            
                                                                 
 bidirectional_2 (Bidirectio  (None, 154, 128)         98816     
 nal)                                                            
                                                                 
 bidirectional_3 (Bidirectio  (None, 154, 128)         98816     
 nal)                                                            
                                                                 
 flatten (Flatten)           (None, 19712)             0

In [92]:
model.fit(X_train,
          y_train,
          validation_data=(X_val, y_val), 
          epochs=65)
# ,        #   batch_size=batch_size)


Epoch 1/65


ValueError: in user code:

    File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\engine\training.py", line 1160, in train_function  *
        return step_function(self, iterator)
    File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\engine\training.py", line 1146, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\engine\training.py", line 1135, in run_step  **
        outputs = model.train_step(data)
    File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\engine\training.py", line 994, in train_step
        loss = self.compute_loss(x, y, y_pred, sample_weight)
    File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\engine\training.py", line 1052, in compute_loss
        return self.compiled_loss(
    File "c:\Users\swast\A

In [ ]:
model.predict(X_val)

1/1 [==============================] - 2s 2s/step


array([[9.67609104e-09, 1.08496904e-13, 3.07748584e-11, 1.04648390e-09,
        4.36772485e-09, 2.82762151e-08, 7.05752335e-03, 4.65633399e-08,
        3.77289155e-07, 1.77266236e-06, 6.18677110e-09, 1.42636802e-03,
        3.60157379e-13, 3.02272720e-14, 1.60717446e-11, 7.64050382e-20,
        1.45653573e-10, 3.38954769e-20, 1.40911220e-08, 5.91647004e-23,
        4.65351917e-08, 7.64199606e-11, 3.06107606e-15, 2.82724015e-03,
        8.38395329e-15, 1.37827274e-15, 1.51483736e-18, 2.09434074e-03,
        4.81686480e-02, 1.35577816e-09, 8.17951758e-18, 2.16874576e-18,
        5.47381806e-25, 1.54129470e-11, 2.45699213e-13, 7.76759954e-03,
        1.44635448e-09, 1.00840081e-03, 3.26304067e-10, 1.46629162e-14,
        6.34178042e-01, 1.43070018e-03, 2.94037789e-01, 5.69382719e-09,
        3.64720596e-13, 1.28914068e-09, 2.25002872e-12, 1.42671438e-13,
        1.05909205e-13, 9.50526044e-07],
       [2.06948542e-07, 3.73569158e-08, 1.82048768e-01, 2.16195033e-07,
        6.56597987e-02,

In [ ]:
# input_shape = ()
# input_shape = (1920, 1080, 3)
# time_steps = 19

# target_height, target_width = 224, 224

# input_layer = tf.zeros((batch_size, time_steps, *input_shape))
# print("Original input shape:", input_layer.shape)

# downsampled_input = downsample_frames(input_layer, target_height, target_width)
# print("Downsampled input shape:", downsampled_input.shape)

# model.build(input_shape=downsampled_input)
